# Bronze Layer - Exploratory Data Analysis

Exploration of data in the `bronze` layer to design the `bronze -> silver` transformation job.

Objectives:
- Detect schema inconsistencies across years (schema drift)
- Define a normalization strategy for data types and column names
- Assess the actual need for deduplication
- Validate the join against the zone reference table (`taxi_zone_lookup`)

## 1. Setup

In [ ]:
from nyc_taxi_lakehouse.spark.session import create_spark_session
from nyc_taxi_lakehouse.config.settings import settings

spark = create_spark_session("bronze_layer_data_exploration")

## 2. Loading sample data

We load a representative month of year (January 2023, 2024 and 2025) to detect potential schema differences between years.

In [ ]:
df_2023 = spark.read.parquet(
    f"s3a://{settings.minio.bucket_name}/{settings.minio.bronze_prefix}/yellow/year=2023/month=01/yellow_tripdata_2023-01.parquet"
)
df_2024 = spark.read.parquet(
    f"s3a://{settings.minio.bucket_name}/{settings.minio.bronze_prefix}/yellow/year=2024/month=01/yellow_tripdata_2024-01.parquet"
)
df_2025 = spark.read.parquet(
    f"s3a://{settings.minio.bucket_name}/{settings.minio.bronze_prefix}/yellow/year=2025/month=01/yellow_tripdata_2025-01.parquet"
)

## 3. Schema inspection and schema drift detection

In [ ]:
df_2023.printSchema()

In [ ]:
df_2024.printSchema()

In [ ]:
df_2025.printSchema()

### Finding: Schema drift across years

| Comparison | Change |
|---|---|
| 2023 → 2024 | `airport_fee` → `Airport_fee` (capitalization change) |
| 2023 → 2024 | `VendorID`, `passenger_count`, `RatecodeID`, `PULocationID`, `DOLocationID`: data type changes (`long`/`double` → more specific types) |
| 2024 → 2025 | Column `cbd_congestion_fee` (double) added — CBD congestion fee effective January 2025 |

**Total columns:** 2023: 21 · 2024: 21 · 2025: 22. 20 columns are common to all three years.

**Conclusion:** The transformation job needs to normalize names and data types, and handle the absence of `cbd_congestion_fee` in years prior to 2025.

## 4. Schema normalization

We apply `normalize_schema()`—the function designed to resolve the schema drift detected above: it renames columns to `snake_case`, casts data types (integers for IDs/categories, `Decimal(10,2)` for amounts), adds `cbd_congestion_fee` when missing, and establishes a canonical column order.

In [ ]:
from nyc_taxi_lakehouse.spark.transformations.schema import normalize_schema

df_2023_norm = normalize_schema(df_2023)
df_2024_norm = normalize_schema(df_2024)
df_2025_norm = normalize_schema(df_2025)

In [ ]:
df_2023_norm.printSchema()

In [ ]:
df_2024_norm.printSchema()

In [ ]:
df_2025_norm.printSchema()

**Result:** The three schemas are identical in terms of column names, types, and order after normalization.

## 5. Duplicate analysis

We assess whether duplicate rows exist in the bronze layer and determine the criteria for handling them in the silver layer.

In [ ]:
total_rows = df_2023_norm.count()
distinct_rows = df_2023_norm.dropDuplicates().count()

print(f"Total: {total_rows}")
print(f"Distinct (exact match): {distinct_rows}")
print(f"Exact duplicates: {total_rows - distinct_rows}")

Zero exact duplicates. We also tested a partial business key (vendor + timestamps + location + rate) to rule out "near-identical" duplicates containing noise in a secondary column.

In [ ]:
business_key = [
    "vendor_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "pu_location_id",
    "do_location_id",
    "fare_amount",
]

total_rows = df_2023_norm.count()
distinct_by_key = df_2023_norm.dropDuplicates(business_key).count()

print(f"Total: {total_rows}")
print(f"Distinct by business key: {distinct_by_key}")
print(f"Duplicates by business key: {total_rows - distinct_by_key}")

In [ ]:
from pyspark.sql import functions as F

duplicated_keys = (
    df_2023_norm
    .groupBy(*business_key)
    .count()
    .filter(F.col("count") > 1)
)

duplicated_keys.show(5)

In [ ]:
df_2023_norm.join(
    duplicated_keys.select(*business_key),
    on=business_key,
    how="inner",
).orderBy(*business_key).show(10, truncate=False)

### Finding: Not duplicates, but financial reversals

The 165 rows identified as "duplicates" based on the partial key are not data loading errors: in each pair, the charges (`mta_tax`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `airport_fee`) have inverted signs and cancel each other out. This is a known NYC TLC pattern representing a transaction and its corresponding reversal or adjustment, rather than an actual duplicate.

**Conclusion:** There are no actual duplicates in the data. The Silver-layer job applies `dropDuplicates()` without arguments as an idempotency safeguard, without additional business logic that would have otherwise removed legitimate transactions.

## 6. Enrichment using the zone lookup table

We map `pu_location_id` / `do_location_id` against `taxi_zone_lookup.csv` to obtain the borough, zone, and service zone for pickup and drop-off.

In [ ]:
from pyspark.sql.functions import col

lookup_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"s3a://{settings.minio.bucket_name}/{settings.minio.reference_prefix}/taxi_zone_lookup.csv")
)

lookup_df_norm = (
    lookup_df
    .withColumnRenamed("LocationID", "location_id")
    .withColumnRenamed("Borough", "borough")
    .withColumnRenamed("Zone", "zone")
    .withColumn("location_id", col("location_id").cast("int"))
)

lookup_df_norm.printSchema()

In [ ]:
pu_lookup = lookup_df_norm.select(
    col("location_id").alias("pu_location_id"),
    col("borough").alias("pu_borough"),
    col("zone").alias("pu_zone"),
    col("service_zone").alias("pu_service_zone"),
)

do_lookup = lookup_df_norm.select(
    col("location_id").alias("do_location_id"),
    col("borough").alias("do_borough"),
    col("zone").alias("do_zone"),
    col("service_zone").alias("do_service_zone"),
)

df_enriched = (
    df_2023_norm
    .join(pu_lookup, on="pu_location_id", how="left")
    .join(do_lookup, on="do_location_id", how="left")
)

df_enriched.select(
    "pu_location_id", "pu_borough", "pu_zone",
    "do_location_id", "do_borough", "do_zone",
).show()

`how="left"` to avoid losing rows where the `location_id` does not match an entry in the lookup table (e.g., "Unknown"/"N/A" codes officially reserved by the NYC TLC).

In [ ]:
unmatched_pu = df_enriched.filter(F.col("pu_borough").isNull()).count()
unmatched_do = df_enriched.filter(F.col("do_borough").isNull()).count()

print(f"Pickup sin match: {unmatched_pu}")
print(f"Dropoff sin match: {unmatched_do}")

**Result:** 0 rows with no match for pickup or drop-off — the lookup table covers 100% of the codes present in the trip data.

## 7. Conclusions

- **Schema drift:** Resolved via `normalize_schema()` — renaming to snake_case, type casting, filling missing columns with `null`, and enforcing a fixed canonical order.
- **Deduplication:** No actual duplicates were detected; `dropDuplicates()` is applied solely as an idempotency safeguard.
- **Enrichment:** `left` join against `taxi_zone_lookup` on `pu_location_id` and `do_location_id`, with 100% coverage.
- These decisions were implemented in `spark/transformations/schema.py` and `spark/transformations/enrichment.py`, and orchestrated in `spark/jobs/transform_silver.py`.

## 8. Silver verification

In [ ]:
df_silver_check = spark.read.parquet(
    f"s3a://{settings.minio.bucket_name}/{settings.minio.silver_prefix}/yellow/year=2023/month=12/yellow_tripdata_2023-12.parquet"
)

df_silver_check.printSchema()
print("Row count:", df_silver_check.count())